# Deepfake Detector MobileNetV3 Small v1

Versao usando **MobileNetV3 Small** com o dataset customizado balanceado do v8.2.

Objetivos desta versao:
- Reduzir tempo de inferencia em relacao ao AlexNet v7/v8.2
- Manter qualidade aceitavel no dataset customizado
- Comparar qualidade e velocidade antes de promover para producao

Dataset usado:
- **Origem**: `data/dataset_colab_v8` preparado localmente no projeto
- **Formato**: `train/real`, `train/fake`, `val/real`, `val/fake`, `test/real`, `test/fake`
- **Balanceamento**: 6000 reais e 6000 fake


In [1]:
# Instalar dependencias
!pip install -q torch torchvision scikit-learn pillow tqdm



/bin/bash: linha 1: pip: comando não encontrado


In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    confusion_matrix,
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')



Usando: cuda


## 1. Download dos Datasets

In [3]:
# Montar Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')



ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Extrair dataset para o disco local do runtime Colab (muito mais rapido que ler do Drive)
DRIVE_ARCHIVE = '/content/drive/MyDrive/dataset_colab_v8.tar.gz'
LOCAL_DATA_ROOT = '/content/data'
DATASET_ROOT = f'{LOCAL_DATA_ROOT}/dataset_colab_v8'

if not os.path.exists(DATASET_ROOT):
    if not os.path.exists(DRIVE_ARCHIVE):
        raise FileNotFoundError(
            f'Arquivo nao encontrado: {DRIVE_ARCHIVE}. '
            'Suba dataset_colab_v8.tar.gz para a raiz do MyDrive.'
        )
    !mkdir -p {LOCAL_DATA_ROOT}
    !tar -xzf {DRIVE_ARCHIVE} -C {LOCAL_DATA_ROOT}

if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f'Dataset nao encontrado em {DATASET_ROOT}.')

print(f'Dataset local encontrado em: {DATASET_ROOT}')


In [ ]:
def collect_images(root_dir, extensions=('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
    """Coleta todos os caminhos de imagem recursivamente."""
    images = []
    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(extensions):
                images.append(os.path.join(root, f))
    return images

train_real = collect_images(os.path.join(DATASET_ROOT, 'train', 'real'))
train_fake = collect_images(os.path.join(DATASET_ROOT, 'train', 'fake'))
val_real = collect_images(os.path.join(DATASET_ROOT, 'val', 'real'))
val_fake = collect_images(os.path.join(DATASET_ROOT, 'val', 'fake'))
test_real = collect_images(os.path.join(DATASET_ROOT, 'test', 'real'))
test_fake = collect_images(os.path.join(DATASET_ROOT, 'test', 'fake'))

print(f'Train real: {len(train_real)}')
print(f'Train fake: {len(train_fake)}')
print(f'Val real: {len(val_real)}')
print(f'Val fake: {len(val_fake)}')
print(f'Test real: {len(test_real)}')
print(f'Test fake: {len(test_fake)}')



In [ ]:
# Construir splits a partir do dataset ja preparado
train_data = [(p, 0) for p in train_real] + [(p, 1) for p in train_fake]
val_data = [(p, 0) for p in val_real] + [(p, 1) for p in val_fake]
test_data = [(p, 0) for p in test_real] + [(p, 1) for p in test_fake]

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

print(f'Train: {len(train_data)} ({sum(x[1] for x in train_data)} fakes)')
print(f'Val: {len(val_data)} ({sum(x[1] for x in val_data)} fakes)')
print(f'Test: {len(test_data)} ({sum(x[1] for x in test_data)} fakes)')



In [ ]:
# Resumo do dataset v8.2
all_images = train_data + val_data + test_data
all_reals = [path for path, label in all_images if label == 0]
all_fakes = [path for path, label in all_images if label == 1]

print(f'Total imagens: {len(all_images)}')
print(f'Total reais: {len(all_reals)}')
print(f'Total fakes: {len(all_fakes)}')
print(f'Ratio fake/real: {len(all_fakes)/len(all_reals):.2f}')



## 2. Dataset e Transforms

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            # Em caso de imagem corrompida, retorna outra
            new_idx = (idx + 1) % len(self.data)
            return self.__getitem__(new_idx)

# Transforms - AlexNet espera 224x224 (mesmo que EfficientNet)
# Augmentation agressivo para melhor generalização
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Criar datasets
train_dataset = DeepfakeDataset(train_data, train_transform)
val_dataset = DeepfakeDataset(val_data, val_transform)
test_dataset = DeepfakeDataset(test_data, val_transform)

print(f"Datasets criados: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
# DataLoaders
# Ajuste para Colab lendo dataset do disco local do runtime
BATCH_SIZE = 32
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
PIN_MEMORY = torch.cuda.is_available()

print(f"BATCH_SIZE={BATCH_SIZE} | NUM_WORKERS={NUM_WORKERS} | PIN_MEMORY={PIN_MEMORY}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)



## 3. Modelo MobileNetV3 Small


In [4]:
class DeepfakeDetectorMobileNetV3Small(nn.Module):
    """Detector de deepfake usando MobileNetV3 Small como backbone."""

    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        if pretrained:
            weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            self.backbone = models.mobilenet_v3_small(weights=weights)
        else:
            self.backbone = models.mobilenet_v3_small(weights=None)

        in_features = self.backbone.classifier[0].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )
        self._initialize_classifier()

    def _initialize_classifier(self):
        for module in self.backbone.classifier.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.backbone(x).squeeze(-1)

    def predict_proba(self, x):
        with torch.no_grad():
            logits = self.forward(x)
            return torch.sigmoid(logits)


model = DeepfakeDetectorMobileNetV3Small(pretrained=True, dropout=0.3).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Modelo MobileNetV3 Small criado:')
print(f'  Total de parametros: {total_params:,}')
print(f'  Parametros treinaveis: {trainable_params:,}')


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /home/wesleiferreira/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|████████████████████████████████████████████████████████| 9.83M/9.83M [00:01<00:00, 9.84MB/s]


Modelo MobileNetV3 Small criado:
  Total de parametros: 1,222,945
  Parametros treinaveis: 1,222,945


## 4. Treinamento

In [5]:
# Focal Loss para lidar com exemplos difíceis
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        focal_weight = (1 - pt) ** self.gamma
        
        # Alpha balancing
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        return (alpha_weight * focal_weight * bce).mean()

criterion = FocalLoss(alpha=0.5, gamma=2.0)

# Optimizer para MobileNetV3 Small
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

In [6]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0
    
    for images, labels in tqdm(loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).long()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    acc = (all_preds == all_labels).mean()
    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, all_preds)
    
    return total_loss / len(loader), acc, auc, f1, all_probs, all_labels

In [7]:
# Treinamento
EPOCHS = 20  # Mantem comparabilidade com AlexNet v8.2
best_auc = 0
best_model_state = None
patience = 12
no_improve = 0

print("="*60)
print("TREINAMENTO - MobileNetV3 Small Deepfake Detector v1")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'='*50}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_auc, val_f1, _, _ = evaluate(model, val_loader)
    
    scheduler.step()
    
    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}, F1: {val_f1:.4f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        best_model_state = model.state_dict().copy()
        print(f"✓ Novo melhor modelo! AUC: {best_auc:.4f}")
        no_improve = 0
    else:
        no_improve += 1
        print(f"Sem melhora ({no_improve}/{patience})")
    
    if no_improve >= patience:
        print("\n⚠ Early stopping!")
        break

# Carregar melhor modelo
model.load_state_dict(best_model_state)
print(f"\n✓ Melhor modelo carregado (AUC: {best_auc:.4f})")



TREINAMENTO - MobileNetV3 Small Deepfake Detector v1

Epoch 1/20 | LR: 3.00e-04


NameError: name 'train_loader' is not defined

## 5. Calibração de Threshold

In [8]:
# Avaliar no conjunto de validação para calibrar threshold
_, _, _, _, val_probs, val_labels = evaluate(model, val_loader)

# Encontrar threshold que maximiza F1
precision, recall, thresholds = precision_recall_curve(val_labels, val_probs)

# Calcular F1 para cada threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

# Threshold conservador (alta precisão para minimizar falsos positivos)
high_precision_idx = np.where(precision >= 0.9)[0]
if len(high_precision_idx) > 0:
    conservative_threshold = thresholds[high_precision_idx[0]] if high_precision_idx[0] < len(thresholds) else 0.7
else:
    conservative_threshold = 0.7

print(f"Threshold ótimo (F1): {best_threshold:.4f}")
print(f"Threshold conservador (precisão ≥90%): {conservative_threshold:.4f}")

# Usar threshold conservador para evitar falsos positivos
final_threshold = conservative_threshold
print(f"\nUsando threshold final: {final_threshold:.4f}")

NameError: name 'val_loader' is not defined

## 6. Avaliação Final

In [ ]:
# Avaliar no conjunto de teste com threshold calibrado
test_loss, test_acc, test_auc, test_f1, test_probs, test_labels = evaluate(model, test_loader)

# Calcular métricas com threshold conservador
test_preds_calibrated = (test_probs > final_threshold).astype(int)

print("\n" + "="*60)
print("RESULTADOS FINAIS - MobileNetV3 Small (Conjunto de Teste)")
print("="*60)
print(f"AUC-ROC: {test_auc:.4f}")
print(f"Threshold: {final_threshold:.4f}")
print("\nCom threshold padrão (0.5):")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"\nCom threshold conservador ({final_threshold:.2f}):")
calibrated_acc = (test_preds_calibrated == test_labels).mean()
calibrated_f1 = f1_score(test_labels, test_preds_calibrated)
print(f"  Accuracy: {calibrated_acc:.4f}")
print(f"  F1-Score: {calibrated_f1:.4f}")

print("\n" + "="*60)
print("RELATÓRIO DETALHADO")
print("="*60)
print(classification_report(test_labels, test_preds_calibrated, target_names=['REAL', 'FAKE']))

In [ ]:
# Análise detalhada das previsões
print("\n" + "="*60)
print("DIAGNÓSTICO DE PREDIÇÕES")
print("="*60)

y_true = np.array([label for _, label in test_data])
y_pred = (test_probs >= final_threshold).astype(int)

cm = confusion_matrix(y_true, y_pred)
print("\nMatriz de confusão:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")
print(f"Predições positivas: {int(y_pred.sum())} de {len(y_pred)} ({100*y_pred.mean():.1f}%)")

real_probs = test_probs[y_true == 0]
fake_probs = test_probs[y_true == 1]

print(f"\nImagens REAIS (n={len(real_probs)}):")
print(f"  Média de p_fake: {real_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(real_probs):.4f}")
print(f"  Min p_fake: {real_probs.min():.4f}")
print(f"  Max p_fake: {real_probs.max():.4f}")
print(f"  Falsos positivos (>{final_threshold:.2f}): {(real_probs > final_threshold).sum()} de {len(real_probs)} ({100*(real_probs > final_threshold).sum()/len(real_probs):.1f}%)")

print(f"\nImagens FAKE (n={len(fake_probs)}):")
print(f"  Média de p_fake: {fake_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(fake_probs):.4f}")
print(f"  Min p_fake: {fake_probs.min():.4f}")
print(f"  Max p_fake: {fake_probs.max():.4f}")
print(f"  Falsos negativos (<{final_threshold:.2f}): {(fake_probs < final_threshold).sum()} de {len(fake_probs)} ({100*(fake_probs < final_threshold).sum()/len(fake_probs):.1f}%)")

plt.figure(figsize=(10, 5))
plt.hist(real_probs, bins=40, alpha=0.6, label='Real', color='tab:blue')
plt.hist(fake_probs, bins=40, alpha=0.6, label='Fake', color='tab:red')
plt.axvline(final_threshold, color='black', linestyle='--', label=f'Threshold={final_threshold:.3f}')
plt.title('Distribuição de scores p_fake no teste')
plt.xlabel('p_fake')
plt.ylabel('Frequência')
plt.legend()
plt.show()



## 7. Comparação com EfficientNet

In [ ]:
# Comparação de características
print('\n' + '='*60)
print('COMPARAÇÃO: MobileNetV3 Small vs AlexNet')
print('='*60)

comparison_data = {
    'Métrica': ['Parâmetros', 'Foco', 'Velocidade Inferência', 'Uso esperado'],
    'MobileNetV3 Small': ['~2.5M', 'Mobile/CPU', 'Mais rápida', 'Candidato leve'],
    'AlexNet': ['~61M', 'Baseline histórico', 'Mais lenta', 'Comparativo v7/v8.2'],
}

print(f"{'Métrica':<25} {'MobileNetV3 Small':<24} {'AlexNet':<24}")
print('-' * 76)
for i in range(len(comparison_data['Métrica'])):
    print(f"{comparison_data['Métrica'][i]:<25} {comparison_data['MobileNetV3 Small'][i]:<24} {comparison_data['AlexNet'][i]:<24}")

print('\n\nResultados deste treinamento (MobileNetV3 Small):')
print(f'  AUC-ROC: {test_auc:.4f}')
print(f'  F1-Score: {calibrated_f1:.4f}')
print(f'  Accuracy: {calibrated_acc:.4f}')


## 8. Salvar Modelo

In [ ]:
# Salvar checkpoint completo
checkpoint = {
    'model_state_dict': best_model_state,
    'best_auc': float(best_auc),
    'best_threshold': float(final_threshold),
    'test_auc': float(test_auc),
    'test_f1': float(calibrated_f1),
    'test_accuracy': float(calibrated_acc),
    'config': {
        'model_name': 'mobilenet_v3_small',
        'version': 'v1',
        'dropout': 0.3,
        'datasets': ['dataset_colab_v8'],
        'description': 'MobileNetV3 Small treinado com dataset_colab_v8 para reduzir tempo de inferencia',
        'input_size': 224,
        'num_features': 576
    }
}

save_path = 'mobilenet_v3_small_deepfake_v1.pth'
torch.save(checkpoint, save_path)
print(f'\n✓ Modelo salvo em: {save_path}')
print(f'  AUC: {best_auc:.4f}')
print(f'  Threshold: {final_threshold:.4f}')
print(f'  Tamanho: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB')


In [ ]:
# Instrucoes para usar o modelo
print("""
╔══════════════════════════════════════════════════════════════════════╗
║          MobileNetV3 Small DEEPFAKE DETECTOR v1 - COMPLETO!          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Para usar na aplicacao DeepTrustShield:                             ║
║                                                                      ║
║  1. Copie o arquivo para a pasta de pesos:                           ║
║     cp mobilenet_v3_small_deepfake_v1.pth model/weights/             ║
║                                                                      ║
║  2. Reinicie o worker para carregar o novo modelo                    ║
║                                                                      ║
║  O modelo MobileNetV3 Small v1 oferece:                              ║
║  - Backbone leve para reduzir tempo de inferencia                    ║
║  - Mesmo dataset customizado balanceado do v8.2                      ║
║  - Comparacao direta com AlexNet v7/v8.2                             ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")
